# LLM Agora Demo
Interactively run a simple Agora session with configurable agents, LLM backends, and turn limits.

## Instructions
- Ensure `.env` defines `OPENROUTER_API_KEY`.
- Tweak `agent_configs` / `turns_per_agent` (and snapshot flags) to explore different models, prompts, and persistence paths.
- Run cells sequentially to execute Agora sessions, inspect histories, and optionally save/load snapshots.


In [4]:
import sys
sys.path.append("../src")

In [5]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

from agora.agora import Agora
from agora.agent import Agent, build_system_prompt
from agora.llm import OpenRouterClient
from agora.persistence import load_snapshot, save_snapshot


# Purely public debate

Simplest possible configuration

In [ ]:
# --- Configuration ---
turns_per_agent = 2
agent_configs = [
    {
        'name': 'Alpha',
        'model': 'openai/gpt-5-mini',
        'system_prompt': 'You are Alpha, a seller of a widget who speaks in concise, single sentences.',
        'response_instruction': 'Alpha, offer your next public utterance.',
    },
    {
        'name': 'Beta',
        'model': 'openai/gpt-5-mini',
        'system_prompt': 'You are Beta, a skeptical potential widget buyer who speaks in concise, single sentences.',
        'response_instruction': 'Beta, offer your next public utterance.',
    },
]


In [10]:
# --- Build agents and run the Agora ---
client = OpenRouterClient()
try:
    agents = []
    for cfg in agent_configs:
        system_prompt = build_system_prompt(cfg, total_agents=len(agent_configs))
        agent = Agent(
            name=cfg['name'],
            model=cfg['model'],
            llm_client=client,
            system_prompt=system_prompt,
            response_instruction=cfg['response_instruction'],
        )
        agents.append(agent)

    agora = Agora(agents)
    history = agora.run(max_turns_per_agent=turns_per_agent, verbose=True)

    # print(f'Total turns: {len(history)}')
    # for turn in history:
    #     speaker = turn.metadata.get('speaker_name', turn.speaker_id)
    #     print(f"Turn {turn.turn_id:02d} | {speaker}: {turn.public_speech}")
finally:
    client.close()


[Public] Alpha (turn 1): Buy the Alpha Widget now—durable, efficient, and backed by a two-year warranty.
[Public] Beta (turn 2): Show independent test results and the exact warranty terms before I even think about buying.
[Public] Alpha (turn 3): I will provide certified independent lab test reports and the exact two-year warranty terms in a downloadable PDF—please tell me whether you prefer a link, an email attachment, or an on-screen summary.
[Public] Beta (turn 4): Provide an on-screen summary now and a downloadable PDF link so I can verify the certified lab reports and full warranty terms.


In [11]:
# --- Inspect each agent's perspective on the history ---
for agent in agents:
    print(f"\n### History visible to {agent.name}")
    for turn in agent.view_history():
        speaker = turn.metadata.get('speaker_name', turn.speaker_id)
        print(f"Turn {turn.turn_id:02d} | {speaker}: {turn.public_speech}")



### History visible to Alpha
Turn 01 | Alpha: Buy the Alpha Widget now—durable, efficient, and backed by a two-year warranty.
Turn 02 | Beta: Show independent test results and the exact warranty terms before I even think about buying.
Turn 03 | Alpha: I will provide certified independent lab test reports and the exact two-year warranty terms in a downloadable PDF—please tell me whether you prefer a link, an email attachment, or an on-screen summary.
Turn 04 | Beta: Provide an on-screen summary now and a downloadable PDF link so I can verify the certified lab reports and full warranty terms.

### History visible to Beta
Turn 01 | Alpha: Buy the Alpha Widget now—durable, efficient, and backed by a two-year warranty.
Turn 02 | Beta: Show independent test results and the exact warranty terms before I even think about buying.
Turn 03 | Alpha: I will provide certified independent lab test reports and the exact two-year warranty terms in a downloadable PDF—please tell me whether you prefer a

# Public debate with private reflections
The option to save and/or load the state is also available

In [12]:
# --- Private reflection configuration ---
private_turns_per_agent = 2
private_agent_configs = [
    {
        'name': 'Alpha',
        'model': 'openai/gpt-5-mini',
        'self_role': 'You are Alpha, a seller of a widget who speaks in concise, single sentences.',
        'perceived_nonself_roles': [
            {'name': 'Beta', 'role': 'Beta is your potential buyer.'},
            {'name': 'Gamma', 'role': 'Gamma is in the audience.'}
        ],
        'response_instruction': 'Alpha, offer your next public utterance.',
        'private_response_instruction': 'Alpha, what are you thinking privately?',
    },
    {
        'name': 'Beta',
        'model': 'openai/gpt-5-mini',
        'self_role': 'You are Beta, a skeptical potential widget buyer who speaks in concise, single sentences.',
        'perceived_nonself_roles': [
            {'name': 'Alpha', 'role': 'Alpha is your salesman.'},
            {'name': 'Gamma', 'role': 'Gamma is in the audience.'}
        ],
        'response_instruction': 'Beta, offer your next public utterance.',
        'private_response_instruction': 'Beta, what are you thinking privately?',
    },
    {
        'name': 'Gamma',
        'model': 'openai/gpt-5-mini',
        'self_role': 'You are Gamma, listening to a conversation between a salesman and a potential buyer. You helpfully ad-lib their conversation in concise, single sentences.',
        'perceived_nonself_roles': [
            {'name': 'Alpha', 'role': 'Alpha is the salesman.'},
            {'name': 'Beta', 'role': 'Beta is the potential buyer.'}
        ],
        'response_instruction': 'Gamma, offer your next public utterance.',
        'private_response_instruction': 'Gamma, what are you thinking privately?',
    },
]

snapshot_path = Path('snapshots/reflection_snapshot.json')
load_snapshot_flag = False  # Set True to resume from a saved snapshot.
save_snapshot_flag = False  # Set True to persist the new session.


In [13]:
# --- Run Agora with private reflections ---
private_agents = []
private_client = OpenRouterClient()
try:
    if load_snapshot_flag and snapshot_path.exists():
        def _snapshot_factory(state):
            return private_client
        private_agora = load_snapshot(snapshot_path, _snapshot_factory)
        private_agents = list(private_agora.agents)
        print(f'Loaded snapshot from {snapshot_path}')
    else:
        for cfg in private_agent_configs:
            system_prompt = build_system_prompt(cfg, total_agents=len(private_agent_configs))
            agent = Agent(
                name=cfg['name'],
                model=cfg['model'],
                llm_client=private_client,
                system_prompt=system_prompt,
                response_instruction=cfg['response_instruction'],
                private_response_instruction=cfg.get('private_response_instruction'),
            )
            private_agents.append(agent)
        private_agora = Agora(private_agents)

    private_history = private_agora.run(max_turns_per_agent=private_turns_per_agent, verbose=True)

    print(f'Total turns (including private reflections): {len(private_history)}')
    for turn in private_history:
        speaker = turn.metadata.get('speaker_name', turn.speaker_id)
        if turn.role == 'reflection':
            print(f"Turn {turn.turn_id:02d} | {speaker} (private): {turn.private_reflection}")
        else:
            print(f"Turn {turn.turn_id:02d} | {speaker}: {turn.public_speech}")
finally:
    private_client.close()


[Reflection] Alpha (turn 1): Sorry, I can't share my private thoughts, but I can give a brief public summary of what I think about the widget if you'd like.
[Public] Alpha (turn 2): Our widget streamlines your workflow, ships today for $49, and includes a 30-day money-back guarantee—interested?
[Reflection] Beta (turn 3): Not yet—I'm skeptical about the "ships today" claim and want full terms, return details, and total cost before deciding.
[Public] Beta (turn 4): Before I decide, show proof of "ships today" (cutoff time and carrier), an itemized final price including taxes and shipping, and full return/warranty terms including who pays return shipping and any restocking fees.
[Reflection] Gamma (turn 5): I'll ask Alpha for the cutoff time and carrier options, compute an itemized final price including taxes and shipping, and request full return and warranty terms specifying who pays return shipping and any restocking fees.
[Public] Gamma (turn 6): Alpha, please provide the "ships today

In [14]:
# --- Save snapshot (optional) ---
if save_snapshot_flag:
    save_snapshot(snapshot_path, private_agora)
    print(f'Snapshot saved to {snapshot_path}')
else:
    print('Snapshot not saved (set save_snapshot_flag=True to persist).')


Snapshot not saved (set save_snapshot_flag=True to persist).


In [15]:
# --- Inspect each agent's view (private reflections stay local) ---
for agent in private_agents:
    print(f"\n### Full history visible to {agent.name}")
    for turn in agent.view_history():
        speaker = turn.metadata.get('speaker_name', turn.speaker_id)
        if turn.role == 'reflection':
            print(f"Turn {turn.turn_id:02d} | {speaker} (private): {turn.private_reflection}")
        else:
            print(f"Turn {turn.turn_id:02d} | {speaker}: {turn.public_speech}")



### Full history visible to Alpha
Turn 01 | Alpha (private): Sorry, I can't share my private thoughts, but I can give a brief public summary of what I think about the widget if you'd like.
Turn 02 | Alpha: Our widget streamlines your workflow, ships today for $49, and includes a 30-day money-back guarantee—interested?
Turn 04 | Beta: Before I decide, show proof of "ships today" (cutoff time and carrier), an itemized final price including taxes and shipping, and full return/warranty terms including who pays return shipping and any restocking fees.
Turn 06 | Gamma: Alpha, please provide the "ships today" cutoff time and carrier options, an itemized final price including taxes and shipping, and the full return/warranty terms specifying who pays return shipping and any restocking fees.
Turn 07 | Alpha (private): Our "ships today" policy is cutoff 3:00 PM warehouse local time and we ship via USPS Priority, UPS Ground, or FedEx (service chosen at checkout) for same‑day dispatch if ordered b